# 03 Debug stay windows

Use this notebook when RICU has later `charttime` values than `openicu_dyn.parquet`.

In [ ]:
from pathlib import Path
import polars as pl

from openicu_yaib.validation import scan_dynamic, stay_windows_from_dyn, compare_dyn_to_icustays
from openicu_yaib.compare import scan_dyn, normalize_reference_columns

In [ ]:
OPENICU_DYN = Path("~/output/openicu_yaib/openicu_dyn.parquet")
ICUSTAYS_CSV = Path("~/physionet.org/files/mimiciv/3.1/icu/icustays.csv.gz")
RICU_STAY_WINDOWS = Path("~/output/openicu_yaib/ricu_stay_windows_miiv.parquet")
RICU_DYN = Path("~/output/openicu_yaib/ricu_dynamic_vars_miiv.parquet")

openicu = scan_dynamic(OPENICU_DYN)

## OpenICU dyn end per stay

In [ ]:
openicu_windows = stay_windows_from_dyn(openicu)
openicu_windows.sort("dyn_end", descending=True).head()

## Compare OpenICU dyn with MIMIC icustays-derived windows

In [ ]:
cmp_icustays = compare_dyn_to_icustays(openicu, ICUSTAYS_CSV, end_rounding="floor")
cmp_icustays.filter(pl.col("stay_id") == 39999858)

## Compare against exported RICU stay_windows if available

In [ ]:
if RICU_STAY_WINDOWS.exists():
    ricu_windows = pl.scan_parquet(RICU_STAY_WINDOWS)
    print(ricu_windows.collect_schema())
    display(ricu_windows.head().collect())
else:
    print("RICU stay-windows parquet not found. Run scripts/export_ricu_stay_windows.R first.")

In [ ]:
# Adjust column names here if your RICU stay_windows export differs.
if RICU_STAY_WINDOWS.exists():
    rw = pl.scan_parquet(RICU_STAY_WINDOWS)
    # Common case: columns stay_id and end, where end is a duration in hours or int.
    cols = rw.collect_schema().names()
    if "end" in cols:
        ricu_end = (
            rw.with_columns(
                (
                    pl.col("end").dt.total_seconds() / 3600
                )
                .cast(pl.Int64)
                .alias("ricu_end")
            )
        )
        joined = (
            openicu_windows.lazy()
            .join(ricu_end, on="stay_id", how="left")
            .with_columns((pl.col("dyn_end") - pl.col("ricu_end")).alias("dyn_minus_ricu_end"))
        )
        display(joined.filter(pl.col("stay_id") == 39999858).collect())
        display(joined.group_by("dyn_minus_ricu_end").agg(pl.len().alias("n")).sort("dyn_minus_ricu_end").collect())

## RICU dynamic values for one stay

In [ ]:
if RICU_DYN.exists():
    ricu_dyn = scan_dyn(RICU_DYN)
    cols = ricu_dyn.collect_schema().names()
    if "time" not in cols and "charttime" in cols:
        ricu_dyn = normalize_reference_columns(ricu_dyn, id_col="stay_id", time_col="charttime")
    display(ricu_dyn.filter(pl.col("stay_id") == 39999858).select(["stay_id", "time", "hr", "crea", "glu", "wbc"]).sort("time").collect())
else:
    print("RICU dynamic parquet not found. Run scripts/export_ricu_dynamic_vars.R first.")